# 04 · STAGING: Transformar datos del SQL Server

**Objetivo:** Tomar datos de raw.invoice_lines (BD destino), descomprimirlos, normalizarlos y guardar en staging.invoice_lines.

**Entrada:** raw.invoice_lines (BD destino - WideWorldImportersDW2)

**Salida:** staging.invoice_lines (BD destino - WideWorldImportersDW2)

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO

engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas:')
print(f'  Servidor: {SERVIDOR}')
print(f'  Origen: {BD_ORIGEN}')
print(f'  Destino: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas:
  Servidor: localhost\SQLEXPRESS02
  Origen: WideWorldImporters
  Destino: WideWorldImportersDW2


## 1. Leer datos crudos de raw.invoice_lines

In [14]:
import pandas as pd
import json

# Leer payloads de raw.invoice_lines (en BD DESTINO)
query_raw = """
SELECT 
    id_raw,
    payload
FROM raw.invoice_lines
ORDER BY id_raw
"""

df_raw = pd.read_sql_query(query_raw, engine_destino)
print(f"Total de registros en raw.invoice_lines: {len(df_raw)}")
if len(df_raw) > 0:
    print(f"Primer payload (primeros 200 caracteres):")
    print(f"  {df_raw.iloc[0]['payload'][:200]}...")

Total de registros en raw.invoice_lines: 100
Primer payload (primeros 200 caracteres):
  {"ExtendedPrice": "2645.0", "InvoiceID": "1", "InvoiceLineID": "1", "LastEditedWhen": "2013-01-01T12:00:00", "Quantity": "10", "StockItemID": "67", "TaxAmount": "345.0", "TaxRate": "15.0", "UnitPrice"...


## 2. Descomprimir payloads JSON

In [15]:
# Descomprimir JSONs
datos_lista = []
errores_descompresion = []

for idx, row in df_raw.iterrows():
    try:
        payload = json.loads(row['payload'])
        payload['id_raw'] = row['id_raw']
        datos_lista.append(payload)
    except Exception as e:
        errores_descompresion.append({
            'id_raw': row['id_raw'],
            'error': str(e)
        })

df_descomprimido = pd.DataFrame(datos_lista)
print(f"Registros descomprimidos exitosamente: {len(df_descomprimido)}")
print(f"Errores de descompresión: {len(errores_descompresion)}")

if len(df_descomprimido) > 0:
    print(f"\nEstructura de datos descomprimidos:")
    print(f"  Columnas: {list(df_descomprimido.columns)}")
    print(f"\nMuestra de datos:")
    display(df_descomprimido.head(3))

Registros descomprimidos exitosamente: 100
Errores de descompresión: 0

Estructura de datos descomprimidos:
  Columnas: ['ExtendedPrice', 'InvoiceID', 'InvoiceLineID', 'LastEditedWhen', 'Quantity', 'StockItemID', 'TaxAmount', 'TaxRate', 'UnitPrice', 'id_raw']

Muestra de datos:


,ExtendedPrice,InvoiceID,InvoiceLineID,LastEditedWhen,Quantity,StockItemID,TaxAmount,TaxRate,UnitPrice,id_raw
0,2645.0,1,1,2013-01-01T12:00:00,10,67,345.0,15.0,-230.0,1
1,134.55,2,2,2013-01-01T12:00:00,9,50,17.55,15.0,13.0,2
2,331.2,2,3,2013-01-01T12:00:00,9,10,43.2,15.0,32.0,3


## 3. Normalizar y transformar datos

In [16]:
# Los datos ya vienen completos en el JSON
df_aceptados = df_descomprimido.copy()

print(f"Registros para STAGING: {len(df_aceptados)}")

# Convertir montos a float
if 'ExtendedPrice' in df_aceptados.columns:
    df_aceptados['ExtendedPrice'] = pd.to_numeric(df_aceptados['ExtendedPrice'], errors='coerce')
if 'TaxAmount' in df_aceptados.columns:
    df_aceptados['TaxAmount'] = pd.to_numeric(df_aceptados['TaxAmount'], errors='coerce')

# Calcular neto = ExtendedPrice - TaxAmount
if 'ExtendedPrice' in df_aceptados.columns and 'TaxAmount' in df_aceptados.columns:
    df_aceptados['neto'] = df_aceptados['ExtendedPrice'] - df_aceptados['TaxAmount']

print(f"\nDatos normalizados:")
display(df_aceptados[['id_raw', 'InvoiceID', 'StockItemID', 'Quantity']].head())

Registros para STAGING: 100

Datos normalizados:


,id_raw,InvoiceID,StockItemID,Quantity
0,1,1,67,10
1,2,2,50,9
2,3,2,10,9
3,4,3,114,3
4,5,4,206,96


## 4. Preparar para guardar en staging

In [17]:
# Seleccionar y renombrar columnas
columnas_mapping = {
    'id_raw': 'id_linea',
    'InvoiceID': 'id_factura',
    'CustomerID': 'id_cliente',
    'StockItemID': 'id_producto',
    'Quantity': 'cantidad',
    'neto': 'neto',
    'TaxAmount': 'impuesto',
    'ExtendedPrice': 'total'
}

# Tomar solo las columnas que existen
cols_existentes = [c for c in columnas_mapping.keys() if c in df_aceptados.columns]
staging_final = df_aceptados[cols_existentes].copy()
staging_final = staging_final.rename(columns={k: v for k, v in columnas_mapping.items() if k in cols_existentes})

print(f"Estructura final para STAGING:")
print(staging_final.dtypes)
print(f"\nMuestra:")
display(staging_final.head())

Estructura final para STAGING:
id_linea         int64
id_factura         str
id_producto        str
cantidad           str
neto           float64
impuesto       float64
total          float64
dtype: object

Muestra:


,id_linea,id_factura,id_producto,cantidad,neto,impuesto,total
0,1,1,67,10,2300.0,345.00,2645.00
1,2,2,50,9,117.0,17.55,134.55
2,3,2,10,9,288.0,43.20,331.20
3,4,3,114,3,90.0,13.50,103.50
4,5,4,206,96,259.2,38.88,298.08


## 5. Crear tabla staging.invoice_lines

In [19]:



create_table_sql = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'staging')
    EXEC('CREATE SCHEMA staging');

IF NOT EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES 
              WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'invoice_lines')
BEGIN
    CREATE TABLE staging.invoice_lines (
        id_linea INT PRIMARY KEY,
        id_factura INT,
        id_cliente INT,
        id_producto INT,
        cantidad INT,
        neto DECIMAL(18, 2),
        impuesto DECIMAL(18, 2),
        total DECIMAL(18, 2),
        timestamp_carga DATETIME DEFAULT GETDATE()
    )
END
"""

try:
    with engine_destino.connect() as conn:
        conn.execute(create_table_sql)
        conn.commit()
        print("Tabla staging.invoice_lines verificada/creada.")
except Exception as e:
        print(f"Error al crear tabla: {e}")

Error al crear tabla: Not an executable object: "\nIF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'staging')\n    EXEC('CREATE SCHEMA staging');\n\nIF NOT EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES \n              WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'invoice_lines')\nBEGIN\n    CREATE TABLE staging.invoice_lines (\n        id_linea INT PRIMARY KEY,\n        id_factura INT,\n        id_cliente INT,\n        id_producto INT,\n        cantidad INT,\n        neto DECIMAL(18, 2),\n        impuesto DECIMAL(18, 2),\n        total DECIMAL(18, 2),\n        timestamp_carga DATETIME DEFAULT GETDATE()\n    )\nEND\n"


## 6. Guardar datos en SQL

In [22]:


# Escribir datos
# Insertar solo las líneas que aún no existen (permite volver a ejecutar la celda)
if len(staging_final) > 0:
    ids_existentes = pd.read_sql_query(
        "SELECT id_linea FROM staging.invoice_lines",
        engine_destino
    )["id_linea"]

    filas_nuevas = (
        staging_final
        .drop_duplicates(subset=["id_linea"])
        .loc[lambda df: ~df["id_linea"].astype(int).isin(ids_existentes)]
    )

    if not filas_nuevas.empty:
        filas_nuevas.to_sql(
            "invoice_lines",
            con=engine_destino,
            schema="staging",
            if_exists="append",
            index=False
        )

    print(
        f"[OK] {len(filas_nuevas)} registros nuevos guardados; "
        f"{len(staging_final) - len(filas_nuevas)} ya existían."
    )
    

[OK] 0 registros nuevos guardados; 100 ya existían.


## 7. Validar datos guardados

In [ ]:
# Verificar datos en SQL
query_validar = """
SELECT 
    COUNT(*) as total_registros,
    COUNT(DISTINCT id_factura) as facturas_unicas,
    SUM(CAST(cantidad AS BIGINT)) as cantidad_total,
    SUM(neto) as neto_total,
    SUM(impuesto) as impuesto_total,
    SUM(total) as total_general
FROM staging.invoice_lines
"""

try:
    df_validar = pd.read_sql_query(query_validar, engine_destino)
    print("Validación de datos en staging.invoice_lines:")
    display(df_validar)
    
    print(f"\nMuestra de datos guardados (primeras 10 filas):")
    query_muestra = "SELECT TOP 10 * FROM staging.invoice_lines ORDER BY id_linea"
    df_muestra = pd.read_sql_query(query_muestra, engine_destino)
    display(df_muestra)
except Exception as e:
        print(f"Error al validar: {e}")